In [4]:
import numpy as np
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import os

# ==========================================
# PHẦN 1: THUẬT TOÁN GAUSS-JORDAN XUẤT MARKDOWN
# ==========================================

def matrix_to_latex_core(np_mat, precision=4):
    """Hàm lõi tạo chuỗi bmatrix thuần (không chứa ký hiệu $$)"""
    mat = np.copy(np_mat).astype(float)
    mat[np.abs(mat) < 1e-10] = 0.0 # Xóa lỗi -0.0
    mat = np.round(mat, precision)
    
    latex_str = [r"\begin{bmatrix}"]
    for row in mat:
        # Dùng format %g để loại bỏ các số 0 thừa
        row_str = " & ".join([f"{val:g}" for val in row])
        latex_str.append("  " + row_str + r" \\")
    latex_str.append(r"\end{bmatrix}")
    return "\n".join(latex_str)

def matrix_to_latex(np_mat, precision=4):
    """Hàm bọc $$ cho ma trận đứng độc lập"""
    return "$$\n" + matrix_to_latex_core(np_mat, precision) + "\n$$"

def run_gauss_jordan(aug_mat, n_cols_b):
    m, total_cols = aug_mat.shape
    n_cols_a = total_cols - n_cols_b
    
    md = []
    md.append("# TRÌNH BÀY LỜI GIẢI GAUSS-JORDAN\n")
    md.append("### 1. Ma trận bổ sung ban đầu $[A|B]$:\n")
    md.append(matrix_to_latex(aug_mat) + "\n")
    
    steps = []
    current_mat = aug_mat.astype(float).copy()
    pivot_row = 0
    pivots = []

    # Quá trình khử Gauss-Jordan
    for j in range(n_cols_a):
        if pivot_row >= m:
            break
            
        # Partial Pivoting
        pivot = np.argmax(np.abs(current_mat[pivot_row:, j])) + pivot_row
        if np.abs(current_mat[pivot, j]) < 1e-10:
            continue
        
        # Đổi chỗ
        if pivot != pivot_row:
            current_mat[[pivot, pivot_row]] = current_mat[[pivot_row, pivot]]
            steps.append(f"Đổi chỗ hàng {pivot+1} và hàng {pivot_row+1}")
            
        # Chuẩn hóa
        div = current_mat[pivot_row, j]
        current_mat[pivot_row] /= div
        pivots.append((pivot_row, j))
        
        # Khử các hàng khác
        for i in range(m):
            if i != pivot_row:
                factor = current_mat[i, j]
                if np.abs(factor) > 1e-10:
                    current_mat[i] -= factor * current_mat[pivot_row]
                    # Thêm ngoặc nhọn {} để LaTeX render đúng chỉ số có 2 chữ số
                    steps.append(f"$L_{{{i+1}}} \\leftarrow L_{{{i+1}}} - ({factor:g}) L_{{{pivot_row+1}}}$")
        
        if len(pivots) == 1:
            md.append(f"### 2. Bước đầu tiên: Cố định Pivot tại cột {j+1}\n")
            md.append(f"**Biến đổi:** {steps[-1] if steps else 'Chuẩn hóa hàng 1'}\n")
            md.append(matrix_to_latex(current_mat) + "\n")
            md.append("### 3. Các bước trung gian (Mô tả biến đổi):\n")
            first_step_idx = len(steps)
        
        pivot_row += 1

    if len(pivots) > 0 and len(steps) > first_step_idx:
        for s in steps[first_step_idx:-1]:
            md.append(f"* {s}")
        md.append("\n")

    md.append("### 4. Ma trận bậc thang rút gọn (RREF):\n")
    if steps and len(steps) > first_step_idx: 
        md.append(f"**Biến đổi cuối:** {steps[-1]}\n")
    md.append(matrix_to_latex(current_mat) + "\n")

    # KIỂM TRA VÔ NGHIỆM
    is_inconsistent = False
    for i in range(m):
        if np.allclose(current_mat[i, :n_cols_a], 0, atol=1e-10) and not np.allclose(current_mat[i, n_cols_a:], 0, atol=1e-10):
            is_inconsistent = True
            break
            
    if is_inconsistent:
        md.append("### 5. Kết luận\n")
        md.append("Tồn tại hàng có dạng $[0\\ 0\\ ...\\ 0\\ |\\ c]$ với $c \\neq 0$.\n")
        md.append("$\\Rightarrow$ Hệ phương trình **VÔ NGHIỆM**.\n")
        return "\n".join(md)

    
    # TÌM NGHIỆM
    pivot_cols = [p[1] for p in pivots]
    free_cols = [c for c in range(n_cols_a) if c not in pivot_cols]
    
    X0 = np.zeros((n_cols_a, n_cols_b))
    for r, c in pivots:
        X0[c, :] = current_mat[r, n_cols_a:]
        
    if not free_cols:
        md.append("### 5. Kết quả: Nghiệm duy nhất\n")
        md.append("Hệ có nghiệm duy nhất $X = $\n")
        md.append(matrix_to_latex(X0) + "\n")
    else:
        basis_vectors = []
        for free_c in free_cols:
            V = np.zeros((n_cols_a, 1))
            V[free_c] = 1
            for r, c in pivots:
                V[c] = -current_mat[r, free_c]
            basis_vectors.append((free_c, V))

        md.append("### 5. Kết quả: Hệ có vô số nghiệm\n")
        
        # --- ĐOẠN ĐƯỢC THÊM MỚI ĐỂ XỬ LÝ MA TRẬN DÒNG ---
        if n_cols_b == 1:
            md.append("Nghiệm tổng quát $X = X_0 + \\sum t_i \\cdot V_i$:\n\n")
        else:
            md.append("Nghiệm tổng quát $X = X_0 + \\sum V_i \\cdot T_i$ (với $T_i$ là ma trận dòng tham số):\n\n")
        
        res_str = "$$\n X = " + matrix_to_latex_core(X0)
        for f_idx, v in basis_vectors:
            if n_cols_b == 1:
                # Nếu B có 1 cột -> Dùng tham số vô hướng t_i ở đằng trước Vector
                param = f"t_{{{f_idx+1}}}"
                res_str += f" + {param} \\cdot " + matrix_to_latex_core(v)
            else:
                # Nếu B có >1 cột -> Dùng ma trận dòng [t_1, t_2] ở đằng sau Vector
                param_elements = [f"t_{{{f_idx+1},{k+1}}}" for k in range(n_cols_b)]
                param_matrix = r"\begin{bmatrix} " + " & ".join(param_elements) + r" \end{bmatrix}"
                res_str += f" + " + matrix_to_latex_core(v) + " " + param_matrix
        res_str += "\n$$"
        
        md.append(res_str + "\n")
        # ------------------------------------------------

    return "\n".join(md)


# ==========================================
# PHẦN 2: GIAO DIỆN TKINTER NHẬP TỪ FILE
# ==========================================

class MatrixApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Công cụ Giải Gauss-Jordan (Hỗ trợ File txt)")
        self.root.geometry("600x500")
        
        config_frame = ttk.LabelFrame(root, text="1. Cấu hình vế phải", padding=10)
        config_frame.pack(fill="x", padx=10, pady=5)
        
        ttk.Label(config_frame, text="Số cột của ma trận B (vế phải):").grid(row=0, column=0, padx=5)
        self.cols_b_var = tk.IntVar(value=1)
        ttk.Entry(config_frame, textvariable=self.cols_b_var, width=5).grid(row=0, column=1)
        
        input_frame = ttk.LabelFrame(root, text="2. Nhập ma trận [A | B]", padding=10)
        input_frame.pack(fill="both", expand=True, padx=10, pady=5)
        
        toolbar = ttk.Frame(input_frame)
        toolbar.pack(fill="x", pady=(0, 5))
        
        ttk.Button(toolbar, text="Mở file Notepad (.txt)", command=self.load_file).pack(side="left")
        ttk.Label(toolbar, text=" Hoặc copy/paste thẳng vào ô bên dưới:", foreground="gray").pack(side="left", padx=10)
        
        self.text_area = tk.Text(input_frame, wrap="none", font=("Consolas", 11))
        self.text_area.pack(fill="both", expand=True)
        
        sample_data = "1 2 3 4\n5 6 7 8\n9 10 11 12"
        self.text_area.insert("1.0", sample_data)
        
        btn_frame = ttk.Frame(root, padding=10)
        btn_frame.pack(fill="x")
        ttk.Button(btn_frame, text="GIẢI & XUẤT FILE MARKDOWN", command=self.solve, style="Accent.TButton").pack(pady=10)

    def load_file(self):
        filepath = filedialog.askopenfilename(filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")])
        if filepath:
            with open(filepath, 'r') as f:
                content = f.read()
            self.text_area.delete("1.0", tk.END)
            self.text_area.insert(tk.END, content)

    def solve(self):
        try:
            n_cols_b = self.cols_b_var.get()
            if n_cols_b <= 0:
                raise ValueError
        except:
            messagebox.showerror("Lỗi", "Số cột của B phải là số nguyên > 0!")
            return
            
        raw_text = self.text_area.get("1.0", tk.END).strip()
        if not raw_text:
            messagebox.showerror("Lỗi", "Dữ liệu ma trận đang trống!")
            return
            
        try:
            import io
            aug_mat = np.loadtxt(io.StringIO(raw_text))
            if aug_mat.ndim == 1: 
                aug_mat = aug_mat.reshape(1, -1)
        except Exception as e:
            messagebox.showerror("Lỗi Cú Pháp", "Không thể đọc ma trận. Hãy đảm bảo dữ liệu chỉ chứa số và được phân cách bằng dấu cách hoặc tab.\nChi tiết: " + str(e))
            return
            
        m, total_cols = aug_mat.shape
        if total_cols <= n_cols_b:
            messagebox.showerror("Lỗi Kích Thước", f"Tổng số cột đọc được ({total_cols}) phải lớn hơn số cột của B ({n_cols_b})!")
            return
            
        save_path = filedialog.asksaveasfilename(
            defaultextension=".md",
            filetypes=[("Markdown Files", "*.md")],
            title="Lưu lời giải vào file"
        )
        
        if not save_path:
            return 
            
        print("\n[HỆ THỐNG] Đang giải và tạo file Markdown...")
        md_content = run_gauss_jordan(aug_mat, n_cols_b)
        
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(md_content)
            
        messagebox.showinfo("Thành công", f"Đã xuất lời giải siêu đẹp ra file:\n{save_path}")

if __name__ == "__main__":
    root = tk.Tk()
    style = ttk.Style()
    style.configure("Accent.TButton", font=("Arial", 10, "bold"), foreground="black")
    app = MatrixApp(root)
    root.mainloop()


[HỆ THỐNG] Đang giải và tạo file Markdown...

[HỆ THỐNG] Đang giải và tạo file Markdown...
